In [18]:
import unicodedata
import numpy as np
import pandas as pd
from keras.src.utils.module_utils import tensorflow
from tensorflow.keras.utils import Sequence
from tensorflow.keras.layers import Conv2D,Dense,Dropout,Input,LSTM,Embedding,MultiHeadAttention,LayerNormalization
import os
from tensorflow.keras.callbacks import EarlyStopping
import datasets
from datasets import Dataset,DatasetDict
import tensorflow as tf
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [19]:
with open ("D:/project_deeplearning/TEP.en-fa.en",encoding="utf-8") as f:
    en_s=f.read().splitlines()

with open ("D:/project_deeplearning/TEP.en-fa.fa",encoding="utf-8") as f:
    fa_s=f.read().splitlines()


assert len(en_s)==len(fa_s)

df=pd.DataFrame(
    {
        "en":en_s,
        "fa":fa_s
    }
)

df=df.sample(n=30000,random_state=42)
df=df.reset_index(drop=True)

en=df["en"]
fa=df["fa"]
data=pd.DataFrame({"train":Dataset.from_pandas(df)})
print(data["train"][0])

{'en': 'stop her . somebody stop her reading .', 'fa': 'متوقفش كنيد يک نفر نگذاره اون ادامه بده .'}


In [20]:
train=data["train"]
train[:10]

0    {'en': 'stop her . somebody stop her reading ....
1        {'en': 'tetrastichous .', 'fa': 'چهاربيتي .'}
2    {'en': 'thats her stage name . i just said tha...
3    {'en': 'i wanna go . what .', 'fa': 'ميخوام بر...
4    {'en': 'im going to see the dragon warrior .',...
5    {'en': 'just think that i've received them and...
6    {'en': 'and the food was no different from a p...
7    {'en': 'are a couple jerkoffs .', 'fa': '2تا ا...
8    {'en': 'i think i should probably just stay wi...
9                {'en': 'vail .', 'fa': 'بکارخوردن .'}
Name: train, dtype: object

In [21]:
def unicode_to_asci(s):
    return "".join(c for c in unicodedata.normalize("NFC",s) if unicodedata.category(c)!='MN' )

In [22]:
len(data)

30000

In [23]:
def preprossing(w):
    w=unicode_to_asci(w.lower().strip())
    w=re.sub(r"([.!?])",r"\1",w)
    w=re.sub(r'([""])+',"",w)
    w=w.rstrip().strip()
    w = "<start> " + w + " <end>"
    return w

In [24]:
en_sen="Im very happy."
preprossing(en_sen)

'<start> im very happy. <end>'

In [25]:
fa_sen="درود بر تو."
preprossing(fa_sen)

'<start> درود بر تو. <end>'

In [26]:
df["en"]=df["en"].apply(preprossing)
df["fa"]=df["fa"].apply(preprossing)

In [27]:
vocab_size=15000
max_length=256
batch_size=20

token_en=Tokenizer(num_words=vocab_size,filters="")
token_fa=Tokenizer(num_words=vocab_size,filters="")



token_en.fit_on_texts(df["en"])
token_fa.fit_on_texts(df["fa"])

en_seq=token_en.texts_to_sequences(df["en"])
fa_seq=token_fa.texts_to_sequences(df["fa"])

en_seq=pad_sequences(en_seq , maxlen=max_length,padding="post")
fa_seq=pad_sequences(fa_seq,maxlen=max_length,padding="post")

decoder_inputs_array=fa_seq[:,:-1]
decoder_targets_array=fa_seq[:,1:]

dataset = tf.data.Dataset.from_tensor_slices(((en_seq, decoder_inputs_array), decoder_targets_array))
dataset = dataset.shuffle(buffer_size=len(en_seq)).batch(batch_size).prefetch(tf.data.AUTOTUNE)



In [28]:
(x,dec_in),y=next(iter(dataset))
print(x.shape,dec_in.shape,y.shape)

(20, 256) (20, 255) (20, 255)


In [29]:
latent_dim=256

encoder_inputs=Input(shape=(max_length,),name="encoder_inputs")
encoder_embedding=Embedding(input_dim=vocab_size,output_dim=latent_dim,mask_zero=True)(encoder_inputs)
encoder_output,state_h,state_c=LSTM(latent_dim,return_state=True,return_sequences=True)(encoder_embedding)



decoder_inputs=Input(shape=(None,),name="decoder_inputs")
decoder_embedding=Embedding(input_dim=vocab_size,output_dim=latent_dim,mask_zero=True,name="decoder_embedding")
decoder_embedd=decoder_embedding(decoder_inputs)
decoder_lstm=LSTM(latent_dim,return_sequences=True,return_state=True,name="decoder_lstm")
decoder_outputs,state_h_dec,state_c_dec=decoder_lstm(decoder_embedd,initial_state=[state_h,state_c])


atn=MultiHeadAttention(num_heads=4,key_dim=64)(query=decoder_outputs,key=encoder_output,value=encoder_output)
attn_out=LayerNormalization()(decoder_outputs+atn)



In [30]:
decoder_dense=Dense(vocab_size,activation="softmax")
decoder_outputs=decoder_dense(attn_out)

In [31]:
model=tf.keras.Model([encoder_inputs,decoder_inputs],decoder_outputs)
model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [32]:
early=EarlyStopping(monitor="val_loss",patience=2,
                   restore_best_weights=True)

In [33]:
splt=int(len(en_seq)*0.9)
train_ds=tf.data.Dataset.from_tensor_slices(((en_seq[:splt],decoder_inputs_array[:splt]),decoder_targets_array[:splt])).shuffle(10000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
val_ds=tf.data.Dataset.from_tensor_slices(((en_seq[splt:],decoder_inputs_array[splt:]),decoder_targets_array[splt:])).batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [ ]:
history=model.fit(train_ds, epochs=25, validation_data=val_ds, callbacks=[early])

Epoch 1/25
   2/1350 ━━━━━━━━━━━━━━━━━━━━ 19:16 858ms/step - accuracy: 0.2419 - loss: 9.0797   

In [ ]:
reverse_fa = {v: k for k, v in token_fa.word_index.items()}

In [ ]:
import pickle
model.save('Translator.keras')
pickle.dump(token_fa,open("token_fa.pkl","wb"))
pickle.dump(token_en,open("token_en.pkl","wb"))
pickle.dump(reverse_fa,open("reverse_fa.pkl","wb"))

In [ ]:
from tensorflow.keras.models import load_model
model=load_model("Translator.keras")

In [ ]:
import pickle

token_fa = pickle.load(open("token_fa.pkl","rb"))
token_en = pickle.load(open("token_en.pkl","rb"))
reverse_fa = pickle.load(open("reverse_fa.pkl","rb"))


In [ ]:
encoder_model=tf.keras.Model(encoder_inputs,[encoder_output,state_h,state_c])

In [ ]:
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
enc_out_input = Input(shape=(max_length,latent_dim))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_emb2 = decoder_embedding(decoder_inputs)

decoder_outputs2, state_h2, state_c2 = decoder_lstm(
    decoder_emb2,
    initial_state=decoder_states_inputs
)

atn2=MultiHeadAttention(num_heads=4,key_dim=64)(query=decoder_outputs2,key=enc_out_input,value=enc_out_input)
attn_out2=LayerNormalization()(decoder_outputs2+atn2)
decoder_outputs2 = decoder_dense(attn_out2)

decoder_model = tf.keras.Model(
    [decoder_inputs,decoder_state_input_h,decoder_state_input_c,enc_out_input],
    [decoder_outputs2, state_h2, state_c2]
)



In [ ]:
def translate(sentence):

    sentence=preprossing(sentence)

    seq=token_en.texts_to_sequences([sentence])
    seq=pad_sequences(seq, maxlen=max_length, padding="post")

    enc_out,h,c=encoder_model.predict(seq)
    target_seq=np.array([[token_fa.word_index["<start>"]]])

    stop=False
    decoded= ""

    while not stop:

        output_tokens, h, c=decoder_model.predict([target_seq,h,c,enc_out])
        sampled_token_index=np.argmax(output_tokens[0, -1, :])
        sampled_word=reverse_fa.get(sampled_token_index, "")

        if sampled_word== "<end>" or len(decoded.split())>max_length:
            stop=True
        else:
            decoded+= " " +sampled_word

        target_seq=np.array([[sampled_token_index]])
        states=[h, c]

    return decoded


In [ ]:
print(translate("I love you"))


In [ ]:
print(translate('you can'))

In [ ]:
import sacrebleu

In [ ]:
print(tf.config.list_physical_devices('GPU'))